# Paso 3 · Mejorar el coste y la relevancia del agente

Buscamos mejoras a dos niveles:

**A) Parámetros de la tool (Tavily):** `max_results`, `chunks_per_source`, `search_depth`, `country`...

**B) El agente:**
1. Un system prompt mejor, para que decida bien cuándo buscar.
2. Una tool "ligera" que solo tiene el parámetro `query`, para gastar menos tokens fijos.
3. Formato compacto del resultado (título + URL + snippet).
4. No inyectar búsquedas infructuosas.
5. Filtrar resultados con score bajo.
6. Un LLM juez que evalúa la relevancia con un esquema Pydantic.
7. Prompt caching.
8. Sin búsquedas en paralelo.

> Código en `src/espadas_langgraph_websearchagent/improvements.py`.

## 0. Configuración

In [ ]:
# Cargamos las claves de API desde el fichero .env (ANTHROPIC_API_KEY y TAVILY_API_KEY).
# Si no están, las pedimos por teclado.
import getpass
import os

from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key:\n")

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

In [ ]:
from espadas_langgraph_websearchagent import improvements as imp
from espadas_langgraph_websearchagent import searchagent
from espadas_langgraph_websearchagent.tavily_api import tavily_api_search
from espadas_langgraph_websearchagent.tokens import count_text_tokens, count_tool_definition_tokens

model = searchagent.create_model()
QUERY = "última versión de Python"

## A. Mejoras a nivel de la tool (parámetros de Tavily)

Hacemos la misma búsqueda con distintas configuraciones y contamos cuántos tokens ocuparía el resultado en el `ToolMessage`. Usamos la API directa de Tavily porque así podemos tocar `chunks_per_source`, que la tool de LangChain no deja cambiar.

In [ ]:
configs = {
    "base (5 res, basic, 3 trozos)":      dict(max_results=5, search_depth="basic"),
    "3 resultados":                        dict(max_results=3, search_depth="basic"),
    "3 res + 2 trozos":                    dict(max_results=3, search_depth="basic", chunks_per_source=2),
    "3 res + 1 trozo":                     dict(max_results=3, search_depth="basic", chunks_per_source=1),
    "3 res + 2 trozos + country=spain":    dict(max_results=3, search_depth="basic", chunks_per_source=2, country="spain"),
    "advanced (5 res)":                    dict(max_results=5, search_depth="advanced"),
    "include_answer + 3 res + 1 trozo":    dict(max_results=3, chunks_per_source=1, include_answer="basic"),
}

responses = {}
print(f"{'configuración':<36} {'tokens str(dict)':>16} {'tokens compacto':>16} {'mejor score':>12}")
for name, params in configs.items():
    response = tavily_api_search(QUERY, **params)
    responses[name] = response
    results = response.get("results", [])
    t_dict = count_text_tokens(model, str(response))
    t_compact = count_text_tokens(model, imp.format_compact(results))
    best = max((r["score"] for r in results), default=0)
    print(f"{name:<36} {t_dict:>16,} {t_compact:>16,} {best:>12.2f}")

In [ ]:
# La respuesta generada por Tavily (include_answer) puede bastar para preguntas sencillas
print(responses["include_answer + 3 res + 1 trozo"].get("answer"))

## B2. Tool "ligera": menos tokens fijos

La tool de LangChain envía un JSON enorme en cada llamada. Comparamos tres versiones:
- la de LangChain por defecto,
- la de LangChain con parámetros ajustados (el schema es el mismo, así que los tokens también),
- nuestra tool ligera: `tavily_search(query)` con una frase de descripción.

In [ ]:
for name, tool in {
    "LangChain por defecto": searchagent.create_default_search_tool(),
    "LangChain ajustada":    imp.create_tuned_langchain_tool(),
    "Tool ligera":           imp.create_slim_search_tool(),
}.items():
    print(f"{name:<24} {count_tool_definition_tokens(model, [tool]):>6,} tokens (incluye los 496 del prompt oculto)")

Este ahorro se paga en **todas** las llamadas: 1 por pregunta si no busca y 2 si busca. En un chatbot con miles de preguntas al día, es el ahorro más importante.

## B3-B5. Formato compacto, búsquedas infructuosas y filtro por score

`make_result_processor()` crea la función que transforma el resultado de Tavily en el texto del `ToolMessage`.

**Ojo con las búsquedas infructuosas:** no se puede "no enviar nada". La API de Anthropic obliga a que cada `tool_call` tenga su `ToolMessage` de respuesta. Lo que hacemos es enviar un mensaje cortito (`NO_RESULTS_MESSAGE`) en vez del resultado completo.

In [ ]:
response = responses["base (5 res, basic, 3 trozos)"]

for name, processor in {
    "base (str(dict))":            searchagent.default_result_processor,
    "compacto":                    imp.make_result_processor(compact=True),
    "compacto + score >= 0.5":     imp.make_result_processor(compact=True, min_score=0.5),
    "compacto + score >= 0.7":     imp.make_result_processor(compact=True, min_score=0.7),
}.items():
    text, _ = processor(response, "¿Cuál es la última versión de Python?")
    print(f"{name:<28} {count_text_tokens(model, text):>6,} tokens")

In [ ]:
# Búsqueda sin resultados: así es la respuesta de Tavily cuando no encuentra nada
empty_response = {"query": "xqzv blorptastic 9981", "results": [], "answer": None, "images": []}
text, _ = imp.make_result_processor()(empty_response, "pregunta de prueba")
print("Lo que recibe el LLM:", text)

**¿Qué umbral de score usar?** Tavily no recomienda un valor fijo. Mira la tabla *score vs relevancia LLM* del Paso 2: el umbral bueno es el que quita los resultados que el juez puntuó bajo sin quitar los buenos. Un umbral muy alto puede dejar al agente sin resultados.

## B6. LLM juez con esquema Pydantic

El juez recibe la pregunta y los resultados numerados y devuelve, para cada uno:

```python
class ResultRelevance(BaseModel):
    indice: int        # número del resultado
    relevancia: int    # 0-10
    explicacion: str   # por qué
```

`with_structured_output()` obliga al LLM a responder exactamente con ese formato. Después solo pasamos al agente los resultados con `relevancia >= 5`.

**Ojo:** el juez es **una llamada extra al LLM**. Solo compensa si los tokens que ahorra en el ToolMessage valen más que lo que cuesta el propio juez.

In [ ]:
from espadas_langgraph_websearchagent.judge import RelevanceReport, judge_relevance
from espadas_langgraph_websearchagent.tokens import estimate_cost

# El esquema JSON que se envía al LLM
print(RelevanceReport.model_json_schema())

judgements, judge_usage = judge_relevance("¿Cuál es la última versión de Python?", response["results"])
for j in judgements:
    print(f"[{j.indice}] {j.relevancia}/10 - {j.explicacion}")

judge_cost = estimate_cost(judge_usage["input_tokens"], judge_usage["output_tokens"])
print(f"\nCoste del juez: {judge_usage['input_tokens']:,} in + {judge_usage['output_tokens']:,} out = {judge_cost:.5f} $")

## Comparación de todas las variantes

Cada variante añade **una** mejora sobre la anterior, así vemos el efecto de cada una por separado:

| Variante | Qué añade |
|---|---|
| `0_base` | agente original |
| `1_prompt_v2` | system prompt mejorado |
| `2_params_tool` | 3 resultados, `search_depth="basic"` fijo, sin búsquedas en paralelo |
| `3_tool_ligera` | tool con solo `query`, `chunks_per_source=2`, `country="spain"` |
| `4_formato_compacto` | título + URL + snippet, y mensaje corto si no hay resultados |
| `5_filtro_score` | quita resultados con score < 0,5 |
| `6_juez_llm` | el LLM juez filtra en vez del score |

> Coste aproximado de esta celda: 7 variantes × 9 preguntas. Con Haiku 4.5 son céntimos, más unos 40 créditos de Tavily.

In [ ]:
from espadas_langgraph_websearchagent.evaluation import evaluate_agent, print_summaries, summarize

variants = imp.create_agent_variants(min_score=0.5, min_relevance=5)

summaries = []
rows_by_variant = {}
for name, agent in variants.items():
    print(f"\n######## {name}")
    rows = evaluate_agent(agent)
    rows_by_variant[name] = rows
    summaries.append(summarize(name, rows))

print()
print_summaries(summaries)

In [ ]:
# Calidad: comparamos las respuestas de la variante base y la última en las preguntas que buscan
from espadas_langgraph_websearchagent.questions import QUESTIONS

for i, (question, should_search) in enumerate(QUESTIONS):
    if not should_search:
        continue
    print("=" * 100)
    print("PREGUNTA:", question)
    for name in ("0_base", "5_filtro_score", "6_juez_llm"):
        row = rows_by_variant[name][i]
        print(f"\n--- {name} (tokens: {row['tokens_totales']:,}) ---")
        print(row["respuesta"][:600])

## B7. Prompt caching

Cómo funciona la caché de Anthropic:
- Funciona por **prefijo**. Si el principio de la petición (en este orden: tools → system → mensajes) es idéntico al de una petición de hace menos de 5 minutos, esa parte se lee de caché y cuesta un **10%** del precio normal.
- La primera vez hay que **escribirla**, y eso cuesta un **25% más**.
- **Mínimo en Claude Haiku 4.5: 4.096 tokens.** Si el prefijo es más corto, no se cachea. No da error: simplemente `cache_creation = 0`.

Nuestra primera llamada tiene ~2.000-2.500 tokens, así que **con una sola pregunta la caché no sirve**. En un **chatbot** el historial crece (cada búsqueda añade ~1.000-3.000 tokens) y supera el mínimo; a partir de ahí, cada pregunta nueva lee de caché todo lo anterior.

Lo probamos con una conversación de 4 preguntas seguidas, con caché y sin caché. Las ejecutamos una detrás de otra para que la caché no caduque (5 minutos).

In [ ]:
from espadas_langgraph_websearchagent.questions import CONVERSATION

def best_agent(use_prompt_cache):
    return searchagent.build_agent(
        tools=[imp.create_slim_search_tool()],
        system_prompt_template=imp.SYSTEM_PROMPT_V2,
        parallel_tool_calls=False,
        result_processor=imp.make_result_processor(compact=True, min_score=0.5),
        use_prompt_cache=use_prompt_cache,
    )

usages_no_cache = imp.run_conversation(best_agent(False), CONVERSATION)
usages_cache = imp.run_conversation(best_agent(True), CONVERSATION)

imp.print_conversation_usage("SIN caché", usages_no_cache)
imp.print_conversation_usage("CON caché", usages_cache)

Cómo leer la tabla:
- **Turno 1:** con caché probablemente no se lee nada. Si el prefijo no llega a 4.096 tokens, tampoco se escribe.
- **Turnos siguientes:** cuando el historial supera los 4.096 tokens, `caché leída` debería crecer y el coste por turno bajar.
- Si `caché escrita` > 0 pero nunca hay `caché leída`, la caché **sale más cara** (+25%), algo que puede pasar con preguntas sueltas.

## Conclusiones del Paso 3

> Completa con los números de la tabla de variantes y de la caché. Lo que esperamos ver y por qué:

| Mejora | Ahorra coste | Mejora relevancia | Comentario |
|---|---|---|---|
| System prompt v2 | – | ✅ | debería arreglar *Mundial 2026* y *tiempo en Sevilla* |
| `max_results=3`, `search_depth` fijo | ✅ | ≈ | evita los 2 créditos de `advanced` |
| Sin búsquedas en paralelo | ✅ | ≈ | evita casos como el de la Honda (45% del total) |
| Tool ligera | ✅✅ | ≈ | ahorra en **todas** las llamadas, también sin búsqueda |
| `chunks_per_source` 3 → 2 | ✅ | ≈ / ❌ | con 1 trozo puede perderse la respuesta (ver Paso 2) |
| Formato compacto | ✅ | ≈ | quita claves vacías del JSON |
| Búsquedas infructuosas | ✅ (poco) | ✅ | evita que el LLM "rellene" con resultados vacíos |
| Filtro por score | ✅ | ✅ / ❌ | depende del umbral; el score no siempre es la relevancia real |
| LLM juez | ❌ / ✅ | ✅ | llamada extra; compensa con resultados largos o con historial |
| Prompt caching | ✅ en chatbot | – | en Haiku 4.5 solo a partir de 4.096 tokens de prefijo |

**Mejor variante:** _(rellenar: la que tenga 9/9 aciertos con menor coste)_